# Data Preprocessing Pipeline

**Dataset:** GoodScents + Leffingwell — `goodscents_jadbio_ready.csv`

**Goal:** Prepare clean, split, and scaled feature matrices ready for multi-label classification.

**Feature families:**
- MACCS keys: 166 binary bits
- Morgan fingerprints: 512 binary bits (radius=2)
- Mordred descriptors: 327 continuous physicochemical descriptors

## Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from skmultilearn.model_selection import iterative_train_test_split
import warnings
warnings.filterwarnings('ignore')

## Step 1 — Load Data

In [ ]:
df = pd.read_csv('goodscents_jadbio_ready.csv', sep=';')

LABEL_COLS = [
    'floral', 'fruity', 'sweet', 'woody', 'green', 'spicy',
    'animal_musk', 'earthy', 'citrus', 'chemical', 'gourmand', 'powdery_amber'
]

fp_cols      = [c for c in df.columns if c.startswith('MACCS_') or c.startswith('morgan_')]
mordred_cols = [c for c in df.columns if c not in LABEL_COLS + ['SMILES'] + fp_cols]

print(f'Total molecules      : {len(df)}')
print(f'MACCS + Morgan cols  : {len(fp_cols)}')
print(f'Mordred cols         : {len(mordred_cols)}')
print(f'Label cols           : {len(LABEL_COLS)}')

## Step 2 — Inspect NaN Values

In [ ]:
X_all = df[fp_cols + mordred_cols].values.astype(float)

nan_rows = np.isnan(X_all).any(axis=1).sum()
nan_cols = np.isnan(X_all).any(axis=0).sum()

nan_cols_fp      = np.isnan(df[fp_cols].values.astype(float)).any(axis=0).sum()
nan_cols_mordred = np.isnan(df[mordred_cols].values.astype(float)).any(axis=0).sum()

nan_row_indices = list(np.where(np.isnan(X_all).any(axis=1))[0])

print(f'Rows with NaN        : {nan_rows}')
print(f'Columns with NaN     : {nan_cols}')
print(f'  - in fingerprints  : {nan_cols_fp}')
print(f'  - in Mordred       : {nan_cols_mordred}')
print(f'NaN row indices      : {nan_row_indices}')
print()
print('SMILES of problematic molecules:')
for idx in nan_row_indices:
    print(f'  row {idx}: {df.iloc[idx]["SMILES"]}')

## Step 3 — Drop NaN Rows

Only 5 molecules (0.1% of the dataset) have NaN values — always the same 5 rows across all 327 Mordred features.
This is a Mordred computation failure on specific molecules (very large or unusual structures), not random missing data.
Given the negligible data loss, we drop these rows entirely rather than imputing.

In [ ]:
df_clean = df.dropna(subset=fp_cols + mordred_cols).reset_index(drop=True)

print(f'Molecules before : {len(df)}')
print(f'Molecules after  : {len(df_clean)}')
print(f'Dropped          : {len(df) - len(df_clean)}')
print(f'NaN remaining    : {df_clean[fp_cols + mordred_cols].isna().sum().sum()}')

## Step 4 — Split into Feature Groups

We separate features into two groups because they require different preprocessing:
- **Fingerprints** (MACCS + Morgan): binary (0/1), no scaling needed
- **Mordred**: continuous physicochemical values on different scales, requires StandardScaler

In [ ]:
X_fp      = df_clean[fp_cols].values.astype(float)
X_mordred = df_clean[mordred_cols].values.astype(float)
y         = df_clean[LABEL_COLS].values

print(f'X_fingerprints shape : {X_fp.shape}')
print(f'X_mordred shape      : {X_mordred.shape}')
print(f'y shape              : {y.shape}')

## Step 5 — Remove Zero-Variance Features

Features with zero variance are identical across all molecules — they carry no information and should be removed.
This is applied independently to each feature group.

Note: this step is safe to do before the train/test split because zero-variance detection uses no statistical learning — it simply checks if a column is constant.

In [ ]:
# Fingerprints
vt_fp = VarianceThreshold(threshold=0)
X_fp = vt_fp.fit_transform(X_fp)
fp_cols_kept = np.array(fp_cols)[vt_fp.get_support()]

print(f'Fingerprints: {len(fp_cols)} -> {X_fp.shape[1]} features '
      f'({len(fp_cols) - X_fp.shape[1]} zero-variance dropped)')

# Mordred
vt_mordred = VarianceThreshold(threshold=0)
X_mordred = vt_mordred.fit_transform(X_mordred)
mordred_cols_kept = np.array(mordred_cols)[vt_mordred.get_support()]

print(f'Mordred     : {len(mordred_cols)} -> {X_mordred.shape[1]} features '
      f'({len(mordred_cols) - X_mordred.shape[1]} zero-variance dropped)')
print(f'\nTotal features after zero-variance removal: {X_fp.shape[1] + X_mordred.shape[1]}')

## Step 6 — Train/Test Split

We use `iterative_train_test_split` from scikit-multilearn instead of sklearn's `train_test_split`.

**Why?** Standard stratified splitting works for single-label problems. For multi-label data with imbalanced labels, `iterative_train_test_split` preserves the positive/negative ratio of **each label independently** in both train and test sets — critical given our class imbalance.

In [ ]:
# iterative_train_test_split requires a single X matrix
# We concatenate temporarily just for the split, then separate again
X_combined = np.hstack([X_fp, X_mordred])
n_fp = X_fp.shape[1]

X_train_comb, y_train, X_test_comb, y_test = iterative_train_test_split(
    X_combined, y, test_size=0.2
)

# Separate back into fingerprints and Mordred
X_fp_train    = X_train_comb[:, :n_fp]
X_mordred_train = X_train_comb[:, n_fp:]
X_fp_test     = X_test_comb[:, :n_fp]
X_mordred_test  = X_test_comb[:, n_fp:]

print(f'Train : {X_fp_train.shape[0]} molecules')
print(f'Test  : {X_fp_test.shape[0]} molecules')
print()
print('Label distribution preserved (% positive):')
print(f'{"Label":<20} {"Full":>8} {"Train":>8} {"Test":>8}')
print('-' * 48)
for i, l in enumerate(LABEL_COLS):
    full  = y[:, i].mean() * 100
    train = y_train[:, i].mean() * 100
    test  = y_test[:, i].mean() * 100
    print(f'{l:<20} {full:>7.1f}% {train:>7.1f}% {test:>7.1f}%')

## Step 7 — Scale Mordred Features

`StandardScaler` transforms each feature to zero mean and unit variance:
$$x_{scaled} = \frac{x - \mu_{train}}{\sigma_{train}}$$

Critically, the scaler is **fitted on train only** and applied to both train and test.
Fitting on the full dataset before splitting would leak test set statistics into preprocessing — an overoptimistic evaluation.

In [ ]:
scaler = StandardScaler()

# Fit on train, transform both
X_mordred_train = scaler.fit_transform(X_mordred_train)
X_mordred_test  = scaler.transform(X_mordred_test)      # same scaler, no refit

print('StandardScaler fitted on train only.')
print(f'Mordred train mean (should be ~0): {X_mordred_train.mean():.4f}')
print(f'Mordred train std  (should be ~1): {X_mordred_train.std():.4f}')
print(f'Mordred test mean  (not forced 0): {X_mordred_test.mean():.4f}')

## Step 8 — Correlation Filter on Mordred (Train Only)

Highly correlated Mordred features (r > 0.95) are redundant — they encode the same information.
For each correlated pair, we drop one feature.

The correlation structure is computed on **train only** and the same column mask is applied to test.

In [ ]:
# Compute correlation matrix on train
corr_matrix = np.corrcoef(X_mordred_train.T)
corr_matrix = np.abs(corr_matrix)

# Find columns to drop
upper_triangle = np.triu(corr_matrix, k=1)
cols_to_drop = set()
rows, cols = np.where(upper_triangle > 0.95)
for r, c in zip(rows, cols):
    if c not in cols_to_drop:
        cols_to_drop.add(c)

cols_to_keep = [i for i in range(X_mordred_train.shape[1]) if i not in cols_to_drop]

# Apply same mask to both train and test
X_mordred_train = X_mordred_train[:, cols_to_keep]
X_mordred_test  = X_mordred_test[:, cols_to_keep]

print(f'Mordred features before correlation filter : {len(cols_to_keep) + len(cols_to_drop)}')
print(f'Features dropped (r > 0.95)               : {len(cols_to_drop)}')
print(f'Mordred features remaining                : {X_mordred_train.shape[1]}')

## Step 9 — Concatenate Final Feature Matrices

Fingerprints (unscaled, binary) and Mordred (scaled, filtered) are concatenated into the final feature matrices.

In [ ]:
X_train = np.hstack([X_fp_train, X_mordred_train])
X_test  = np.hstack([X_fp_test,  X_mordred_test])

print('Final feature matrices:')
print(f'  X_train : {X_train.shape}')
print(f'  X_test  : {X_test.shape}')
print(f'  y_train : {y_train.shape}')
print(f'  y_test  : {y_test.shape}')
print()
print('Preprocessing complete. Ready for model training.')